# Session 8 — Tool Use and Function Calling

LLMs are excellent at reasoning but cannot act on the world directly. Tool use (also called function calling) gives the model a way to request external actions — fetching live data, running calculations, searching a document store — and incorporate the results into its answer.

This notebook covers:
1. Anatomy of a tool definition (JSON schema)
2. A custom weather tool — the classic first example
3. The agentic loop step by step
4. Built-in tools: `web_search` and `code_interpreter`
5. Parallel tool calls — two tools in one turn
6. RAG as a tool call — the model decides when to retrieve

## Learning Goals

- understand the JSON schema format for tool definitions
- implement the four-step agentic loop (request → detect → execute → feed back)
- use built-in OpenAI tools without writing a Python function
- define multiple tools and handle parallel calls
- refactor the RAG pipeline from notebook 04 so the model controls retrieval

In [ ]:
import json
import os
import requests
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")

print("OpenAI key present:", bool(OPENAI_API_KEY))


def make_openai_client(api_key=None, base_url=None):
    kwargs = {}
    if api_key:
        kwargs["api_key"] = api_key
    if base_url:
        kwargs["base_url"] = base_url
    if api_key == OPENAI_API_KEY and not base_url:
        if OPENAI_ORG_ID:
            kwargs["organization"] = OPENAI_ORG_ID
        if OPENAI_PROJECT_ID:
            kwargs["project"] = OPENAI_PROJECT_ID
    return OpenAI(**kwargs)


def require_env(name: str):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def session8_dir() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    candidate = cwd / "material" / "Session 8"
    if candidate.exists():
        return candidate
    return cwd

## 1. Anatomy of a Tool Definition

Every tool is a Python dict that follows the OpenAI JSON schema format. The model reads `description` to decide when to call the tool, and `parameters` to know what arguments to pass.

In [ ]:
# Annotated tool schema — nothing executes here, just for reading

weather_tool = {
    "type": "function",           # always "function" for custom tools
    "name": "get_weather",        # must match the Python function name you will call
    "description": (
        "Returns current weather conditions for a city. "
        "Call this whenever the user asks about weather or temperature."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. Sydney, Paris, New York",
            },
            "units": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit to return.",
            },
        },
        "required": ["location", "units"],
        "additionalProperties": False,
    },
    "strict": True,   # model must fill all required fields; no extra fields allowed
}

print(json.dumps(weather_tool, indent=2))

## 2. Custom Tool: Weather

`get_weather` calls the free wttr.in API — no API key needed. The function signature must match the `name` and `parameters` in the schema above.

In [ ]:
def get_weather(location: str, units: str = "celsius") -> dict:
    """Fetch current weather from wttr.in (free, no API key required)."""
    unit_char = "m" if units == "celsius" else "u"
    url = f"https://wttr.in/{requests.utils.quote(location)}?format=j1&{unit_char}"
    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        current = data["current_condition"][0]
        temp = current["temp_C"] if units == "celsius" else current["temp_F"]
        desc = current["weatherDesc"][0]["value"]
        return {
            "location": location,
            "temperature": int(temp),
            "units": units,
            "description": desc,
        }
    except Exception as e:
        return {"error": str(e), "location": location}


# Quick smoke test — confirm the function works before wiring it to the model
result = get_weather("Sydney", "celsius")
print(result)